# Exploratory Data Analysis - Brent Oil Prices

This notebook performs initial exploratory data analysis on Brent oil price data to understand the time series properties and inform modeling choices.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.tsa.seasonal import seasonal_decompose
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

## 1. Data Loading and Initial Inspection

In [ ]:
# Load the Brent oil price data
df = pd.read_csv('../data/BrentOilPrices.csv')

# Convert date column to datetime
df['date'] = pd.to_datetime(df['Date'])
df['price'] = df['Price']
df.set_index('date', inplace=True)

# Sort by date
df.sort_index(inplace=True)

# Display basic information
print("Dataset Shape:", df.shape)
print("\nDate Range:", df.index.min(), "to", df.index.max())
print("\nBasic Statistics:")
print(df.describe())

# Display first few rows
print("\nFirst 5 rows:")
print(df.head())

## 2. Time Series Visualization

In [ ]:
# Plot the raw price series
plt.figure(figsize=(14, 6))
plt.plot(df.index, df['price'], linewidth=2)
plt.title('Brent Oil Prices Over Time', fontsize=16, fontweight='bold')
plt.xlabel('Date', fontsize=12)
plt.ylabel('Price (USD per barrel)', fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../plots/price_series.png', dpi=300, bbox_inches='tight')
plt.show()

## 3. Log Returns Analysis

Log returns are commonly used in financial time series analysis as they help achieve stationarity and normalize price changes.

In [ ]:
# Calculate log returns
df['log_returns'] = np.log(df['price'] / df['price'].shift(1))

# Remove NaN values
df_returns = df.dropna()

# Plot log returns
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Plot 1: Log returns over time
axes[0].plot(df_returns.index, df_returns['log_returns'], linewidth=1, alpha=0.7)
axes[0].axhline(y=0, color='r', linestyle='--', linewidth=1)
axes[0].set_title('Log Returns of Brent Oil Prices', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Date', fontsize=12)
axes[0].set_ylabel('Log Returns', fontsize=12)
axes[0].grid(True, alpha=0.3)

# Plot 2: Distribution of log returns
axes[1].hist(df_returns['log_returns'], bins=30, edgecolor='black', alpha=0.7)
axes[1].axvline(x=df_returns['log_returns'].mean(), color='r', linestyle='--', linewidth=2, label=f'Mean: {df_returns["log_returns"].mean():.4f}')
axes[1].set_title('Distribution of Log Returns', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Log Returns', fontsize=12)
axes[1].set_ylabel('Frequency', fontsize=12)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../plots/log_returns_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

# Print statistics for log returns
print("Log Returns Statistics:")
print(df_returns['log_returns'].describe())
print(f"\nSkewness: {df_returns['log_returns'].skew():.4f}")
print(f"Kurtosis: {df_returns['log_returns'].kurtosis():.4f}")

## 4. Stationarity Testing

We'll perform Augmented Dickey-Fuller (ADF) and KPSS tests to assess stationarity.

In [ ]:
def adf_test(series):
    """
    Perform Augmented Dickey-Fuller test
    H0: Series has unit root (non-stationary)
    H1: Series is stationary
    """
    result = adfuller(series.dropna())
    print('Augmented Dickey-Fuller Test:')
    print(f'ADF Statistic: {result[0]:.4f}')
    print(f'p-value: {result[1]:.4f}')
    print(f'Critical Values:')
    for key, value in result[4].items():
        print(f'\t{key}: {value:.4f}')
    
    if result[1] < 0.05:
        print('Result: Reject H0 - Series is stationary')
    else:
        print('Result: Fail to reject H0 - Series is non-stationary')
    print()

def kpss_test(series):
    """
    Perform KPSS test
    H0: Series is stationary
    H1: Series has unit root (non-stationary)
    """
    result = kpss(series.dropna())
    print('KPSS Test:')
    print(f'KPSS Statistic: {result[0]:.4f}')
    print(f'p-value: {result[1]:.4f}')
    print(f'Critical Values:')
    for key, value in result[3].items():
        print(f'\t{key}: {value:.4f}')
    
    if result[1] < 0.05:
        print('Result: Reject H0 - Series is non-stationary')
    else:
        print('Result: Fail to reject H0 - Series is stationary')
    print()

# Test on raw prices
print("=== Raw Price Series ===")
adf_test(df['price'])
kpss_test(df['price'])

# Test on log returns
print("=== Log Returns Series ===")
adf_test(df_returns['log_returns'])
kpss_test(df_returns['log_returns'])

## 5. Volatility Analysis

Examine volatility patterns and clustering in the returns.

In [ ]:
# Calculate rolling volatility (standard deviation)
window = 30  # 30-day rolling window for volatility
df_returns['rolling_volatility'] = df_returns['log_returns'].rolling(window=window).std()

# Plot volatility
plt.figure(figsize=(14, 6))
plt.plot(df_returns.index, df_returns['rolling_volatility'], linewidth=2, color='orange')
plt.title(f'Rolling Volatility (Window = {window} days)', fontsize=16, fontweight='bold')
plt.xlabel('Date', fontsize=12)
plt.ylabel('Volatility (Std Dev)', fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../plots/volatility_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

# Calculate and print volatility statistics
print("Volatility Statistics:")
print(df_returns['rolling_volatility'].describe())

## 6. Trend and Seasonal Decomposition

Decompose the time series into trend, seasonal, and residual components.

In [ ]:
# Perform seasonal decomposition
# Note: Using annual period for decomposition
decomposition = seasonal_decompose(df['price'], model='additive', period=252)

# Plot decomposition
fig, axes = plt.subplots(4, 1, figsize=(14, 12))

axes[0].plot(df.index, df['price'], linewidth=2)
axes[0].set_title('Original Series', fontsize=12, fontweight='bold')
axes[0].grid(True, alpha=0.3)

axes[1].plot(decomposition.trend.index, decomposition.trend, linewidth=2, color='green')
axes[1].set_title('Trend Component', fontsize=12, fontweight='bold')
axes[1].grid(True, alpha=0.3)

axes[2].plot(decomposition.seasonal.index, decomposition.seasonal, linewidth=2, color='orange')
axes[2].set_title('Seasonal Component', fontsize=12, fontweight='bold')
axes[2].grid(True, alpha=0.3)

axes[3].plot(decomposition.resid.index, decomposition.resid, linewidth=2, color='red')
axes[3].set_title('Residual Component', fontsize=12, fontweight='bold')
axes[3].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../plots/decomposition.png', dpi=300, bbox_inches='tight')
plt.show()

## 7. Key Events Overlay

Visualize how key events relate to price movements.

In [ ]:
# Load key events data
events_df = pd.read_csv('../data/key_events_oil_prices.csv')
events_df['date'] = pd.to_datetime(events_df['date'])

# Plot price series with events
plt.figure(figsize=(16, 8))
plt.plot(df.index, df['price'], linewidth=2, label='Brent Oil Price', alpha=0.7)

# Add vertical lines for events
for idx, event in events_df.iterrows():
    if event['date'] >= df.index.min() and event['date'] <= df.index.max():
        color = 'red' if event['expected_impact'] == 'Positive' else 'blue'
        plt.axvline(x=event['date'], color=color, linestyle='--', alpha=0.5, linewidth=1)
        plt.text(event['date'], df['price'].max()*0.9, event['event_description'][:20] + '...', 
                rotation=90, fontsize=8, alpha=0.7)

plt.title('Brent Oil Prices with Key Events', fontsize=16, fontweight='bold')
plt.xlabel('Date', fontsize=12)
plt.ylabel('Price (USD per barrel)', fontsize=12)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../plots/prices_with_events.png', dpi=300, bbox_inches='tight')
plt.show()

# Display events within the data range
print("Key Events within Data Range:")
events_in_range = events_df[(events_df['date'] >= df.index.min()) & (events_df['date'] <= df.index.max())]
print(events_in_range[['date', 'event_description', 'event_type', 'expected_impact']])

## 8. Summary Statistics and Findings

In [ ]:
# Calculate key statistics
print("=== EDA SUMMARY ===")
print(f"\nData Period: {df.index.min()} to {df.index.max()}")
print(f"Total Observations: {len(df)}")
print(f"\nPrice Statistics:")
print(f"  Mean: ${df['price'].mean():.2f}")
print(f"  Median: ${df['price'].median():.2f}")
print(f"  Std Dev: ${df['price'].std():.2f}")
print(f"  Min: ${df['price'].min():.2f}")
print(f"  Max: ${df['price'].max():.2f}")
print(f"  Total Change: ${df['price'].iloc[-1] - df['price'].iloc[0]:.2f} ({((df['price'].iloc[-1] / df['price'].iloc[0]) - 1) * 100:.1f}%)")

print(f"\nVolatility Statistics:")
print(f"  Mean Daily Volatility: {df_returns['log_returns'].std():.4f}")
print(f"  Max Daily Gain: {df_returns['log_returns'].max():.4f}")
print(f"  Max Daily Loss: {df_returns['log_returns'].min():.4f}")

print(f"\nKey Events in Dataset: {len(events_df)}")
print(f"Events within price data range: {len(events_in_range)}")

## Key Findings:

1. **Data Coverage**: The dataset spans [start date] to [end date] with [number] observations.

2. **Price Trends**: The data shows [describe overall trend - upward/downward/volatile].

3. **Stationarity**: 
   - Raw price series: [stationary/non-stationary] based on ADF/KPSS tests
   - Log returns: [stationary/non-stationary] based on ADF/KPSS tests
   - This suggests [implications for modeling]

4. **Volatility Patterns**: 
   - Volatility clustering is [present/absent]
   - High volatility periods correspond to [events/periods]

5. **Event Correlation**: 
   - [number] key events fall within the data range
   - Visual inspection suggests [correlations/patterns]

6. **Modeling Implications**:
   - Log returns transformation [recommended/not recommended] for change point analysis
   - Volatility modeling may be [important/less important]
   - Event correlation analysis should focus on [time periods/event types]

## Next Steps:

1. Apply Bayesian change point detection to identify structural breaks
2. Compare detected change points with known event dates
3. Quantify the magnitude of price changes around change points
4. Develop probabilistic statements about event impacts